# Machine Learning. EDA To Model Training

### 1. Dataset Overview Titanic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset Loading
df = pd.read_csv("train.csv")

# Head
print("Head")
print(df.head())

# Shape
print("Shape")
print(f"Shape of the data-{df.shape}")

# Info
print(df.info())

# Description
print(df.isnull().sum())

# Drop the missing the values in the Column = "Embarked"
complete = df.drop(columns=["Cabin"])

print(f"\nDropped the missing value\n{complete.isnull().sum()}")

# Divived the dataset into X and Y
x = complete.drop(columns=["Survived"])
y = complete["Survived"]

### 2. Visualisation of the OG Dataset

In [ ]:
gender_color = {
    "male" : "#F0943F",
    "female" : "#12E98F"
}

survival_color = {
     0 : "#F0943F",
     1 : "#12E98F"
}

sns.set_style("dark")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(
    "Statistical Visualization (Titanic)",
    fontsize=25,
    fontweight="bold", 
    y = 1.02
)


#SURVIVED
ax= sns.countplot(
    data = df,
    x = 'Survived',
    hue = 'Sex',
    palette = gender_color,
    ax = axes[0, 0]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

axes[0, 0].set_title("Survival Count By Sex")
axes[0, 0].set_xlabel("Survived 0 = No 1 = Yes")
axes[0, 0].set_ylabel("Count")

#PIE CHARTS
gender = df["Sex"].value_counts()
explode = (0.05, 0.05)

axes[0, 1].pie(
    gender,
    labels = gender.index,
    autopct = "%1.1f%%",
    explode = (0.05, 0.05),
    colors = [gender_color[s] for s in gender.index],
    startangle=90
)

axes[0, 1].set_title("Gender Distribution")

#HISTOGRAM
sns.histplot(
    data = df,
    x = "Age",
    bins = 8,
    hue = "Sex",
    kde = True,
    palette=gender_color,
    ax = axes[0, 2]
)

axes[0, 2].set_title("Age Distribution By Sex")


#COUNTPLOT
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
labels = ["0-10", "11-20", "21-30", "31-40", "41-50", "51-60", "61-70", "71-80"]

df["AgeRange"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels
)


sns.countplot(
    data = df,
    x = "AgeRange",
    hue= "Survived",
    palette = survival_color,
    ax = axes[1, 0]
)

axes[1, 0].set_title("Survival Count By Age")
axes[1, 0].set_xlabel("Age Range")
axes[1, 0].set_ylabel("Survival Count")


#COUNTPLOT
ax= sns.countplot(
    data = df,
    x = 'Pclass',
    hue = 'Survived',
    palette = survival_color,
    ax = axes[1, 1]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

axes[1, 1].set_title("Survival Distribution By Class")
axes[1, 1].set_xlabel("Class Of Passenger")
axes[1, 1].set_ylabel(" Survival Count")



#CORRELATION HEATMAP
numeric_data = df.select_dtypes(include="number")

correlation = numeric_data.corr()

sns.heatmap(
    data=correlation,
    annot=True,
    cmap="RdYlGn",
    ax=axes[1, 2]
)

axes[1, 2].set_title("Correlation Heatmap")


plt.tight_layout()
plt.show()

### 3. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y,
)

### 4. Handling Missing Values

In [ ]:
from sklearn.impute import SimpleImputer

x_train = x_train.copy()
x_test = x_test.copy()

num_impute_cols = ["Age"]
cat_impute_cols = ["Embarked"]

# Filling missing values with median, column = "Age"
num_imputer = SimpleImputer(strategy="median")
x_train[num_impute_cols] = num_imputer.fit_transform(x_train[num_impute_cols])
x_test[num_impute_cols] = num_imputer.transform(x_test[num_impute_cols])

# Filling missing values with most frequent, column = "Embarked"
cat_imputer = SimpleImputer(strategy="most_frequent")
x_train[cat_impute_cols] = cat_imputer.fit_transform(x_train[cat_impute_cols])
x_test[cat_impute_cols] = cat_imputer.transform(x_test[cat_impute_cols])

print(x_train.isnull().sum())

### 5. Scaling Numeric Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

scaler = StandardScaler()

# Scaling the numerical columns 
x_train_scaled = scaler.fit_transform(x_train[scale_cols])
x_test_scaled = scaler.transform(x_test[scale_cols])

print(f"Raw x_train =\n{x_train}")
print("Pclass mean (raw):", x_train["Pclass"].mean().round(2))
print("Pclass mean (scaled):", x_train_scaled[:, 0].mean().round(2))
print("Fare mean (scaled):", x_train_scaled[:, 4].mean().round(2))
print("Parch mean, test (scaled) - should not be exactly 0:", x_test_scaled[:, 3].mean().round(4))

### 6. Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

categorical_cols = ["Sex", "Embarked"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
])

prep_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

X_train_processed = prep_pipeline.fit_transform(x_train, y_train)
X_test_processed = prep_pipeline.transform(x_test)

feature_names = prep_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("\nFeature names:", feature_names)
print("Total features:", len(feature_names))

### 7. Creating Classifier Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))